# Experiment: Emotion Analysis

Objective: estimate coarse emotion categories from review text and compare
emotion trends with binary sentiment labels.


## 1. Setup


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.json"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.core import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


## 4. Data Validation


In [ ]:
assert train_df[LABEL_COL].isin([0, 1]).all()
assert train_df[REVIEW_COL].str.len().gt(0).all()


## 5. Exploratory Analysis


In [ ]:
emotion_lexicon = {
    "joy": {"love", "great", "excellent", "happy", "amazing", "perfect", "wonderful", "best"},
    "anger": {"hate", "angry", "terrible", "awful", "worst", "annoying", "bad"},
    "sadness": {"sad", "disappointed", "upset", "poor", "broken"},
    "fear": {"afraid", "worried", "scared", "risk", "unsafe"},
    "surprise": {"surprised", "unexpected", "suddenly"},
    "disgust": {"disgusting", "gross", "nasty"},
}

emotion_lexicon


## 6. Feature Engineering


In [ ]:
import re


def tokenize(text: str) -> list[str]:
    return re.findall(r"[A-Za-z0-9']+", text.lower())


def score_emotions(text: str, lexicon: dict[str, set[str]]) -> dict[str, int]:
    toks = tokenize(text)
    scores = {emo: 0 for emo in lexicon}
    for tok in toks:
        for emo, words in lexicon.items():
            if tok in words:
                scores[emo] += 1
    return scores

emotion_scores = train_df[REVIEW_COL].apply(lambda t: score_emotions(t, emotion_lexicon))
emotion_df = pd.DataFrame(list(emotion_scores))
train_emotion = pd.concat([train_df[[REVIEW_COL, LABEL_COL]], emotion_df], axis=1)
train_emotion.head()


## 7. Modeling


In [ ]:
emotion_cols = list(emotion_lexicon.keys())
train_emotion["dominant_emotion"] = train_emotion[emotion_cols].idxmax(axis=1)
train_emotion.loc[train_emotion[emotion_cols].sum(axis=1) == 0, "dominant_emotion"] = "none"

emotion_sentiment_table = pd.crosstab(
    train_emotion["dominant_emotion"],
    train_emotion[LABEL_COL],
    normalize="index",
).rename(columns={0: "neg_ratio", 1: "pos_ratio"})
emotion_sentiment_table


## 8. Evaluation


In [ ]:
plot_df = emotion_sentiment_table.reset_index()
ax = sns.barplot(data=plot_df, x="dominant_emotion", y="pos_ratio")
ax.set_ylim(0, 1)
ax.set_title("Positive ratio by dominant emotion")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 9. Inference / Submission


In [ ]:
out = PROCESSED_DIR / "emotion_analysis_summary.csv"
emotion_sentiment_table.reset_index().to_csv(out, index=False)
print(out.relative_to(PROJECT_ROOT))
emotion_sentiment_table


## 10. Conclusions / Next Steps

- Emotion analysis complements sentiment by adding affect categories.
- Here we used a lightweight lexicon approach as a baseline.
- For higher quality, fine-tune an emotion-specific transformer model and compare against this baseline.
